# 11 — Diagnóstico dos modelos e validação externa nova

Comparação do baseline e do candidato da execução 10, **sem treinamento**,
com as mesmas 12 classes, política V2 preservada e confiança **0,25**.

1. Conferir pesos, entradas, separação atual Home e evidência do treino original.
2. Criar uma validação externa de **1.000 imagens** do split oficial
   validation do Open Images, fora dos 100 mil IDs anteriores.
3. Separar outras **300 imagens como reserva futura**, sem inferência.
4. Comparar os modelos na validação nova e na Home VAL histórica,
   com contagens, intervalos pareados e exemplos visuais dos erros.

A validação externa tem anotações parciais e não garante ambientes internos.
A revisão de cenas reais próximas do uso pretendido continua necessária.
Os alertas são comparados com V2 nas caixas; não há validação independente
de risco físico ou de benefício para locomoção. Nenhum peso é substituído.


## 1. Configurar o Kaggle

- Ligue **Internet**.
- GPU T4 é recomendada; **CPU também funciona** para esta inferência.
- Em **Add-ons → Secrets**, habilite o Secret já utilizado:
  **GDRIVE_RCLONE_CONFIG_B64**.
- Use **Save Version → Save & Run All**.

Os caminhos da execução 10 estão preenchidos. Os pesos e as entradas
serão recuperados do Drive. Os 100 lotes anteriores não são copiados.


In [ ]:
%pip install -q ultralytics==8.4.150 pyyaml pandas pillow scipy


In [ ]:
import os
os.environ["YOLO_OFFLINE"] = "true"
import sys, json, csv, shutil, platform, hashlib, inspect, subprocess
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch, ultralytics
from ultralytics import YOLO, settings
from ultralytics.utils.metrics import ap_per_class
from IPython.display import display, FileLink

VERSAO = "11-diagnostico-validacao-independente-1.0"
PASTA_EXECUCAO_10 = "resultados/yolov8n_home_openimages_80_20_1ep_kaggle_20261015_030228_068010"
N_VALIDACAO, N_RESERVA_FUTURA = 1000, 300
SOBRESSALENTES_DOWNLOAD, WORKERS_DOWNLOAD, SEED = 350, 8, 42
DEVICE = 0 if torch.cuda.is_available() else "cpu"
NOME_EXPERIMENTO = "diagnostico11_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
PASTA_LOCAL = Path("/tmp/tcc_diagnostico_11") / NOME_EXPERIMENTO
PASTA_RESULTADOS = Path("/kaggle/working/resultados_tcc") / NOME_EXPERIMENTO
PASTA_LOCAL.mkdir(parents=True, exist_ok=True)
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
if ultralytics.__version__ != "8.4.150":
    raise RuntimeError("Reinicie o ambiente e execute com ultralytics==8.4.150.")
if min(N_VALIDACAO, N_RESERVA_FUTURA) <= 0:
    raise ValueError("Os dois conjuntos precisam ter tamanho positivo.")
settings.update({k: False for k in
    ["sync", "hub", "platform", "wandb", "comet", "mlflow",
     "neptune", "clearml", "dvc", "tensorboard"] if k in settings})
print("Dispositivo:", torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU")
print("Validação nova:", N_VALIDACAO, "| reserva sem inferência:", N_RESERVA_FUTURA)
print("Esta execução não treina nem promove modelos.")


In [ ]:
EXPECTED = {'baseline': '4170653e0c0a604f85b6747e42b3fd2ff5ddb765ee4762915dfc9f3d33d322d9', 'candidato': '97040f91876d8835a67a7db287d3e1c5f129a4068e36dd4483d947f62b260c5f', 'politica': '1aa6b13cb68d8f30a2f388c3577a61a12dcac4c604581877091d3fbdc6042845', 'home_zip': 'ada7fbee37b83f673b1b76bc57643d1fe4648941e0e8a7b895282216aed47676', 'test_home': 'f48be0a2b3a93e59259b1ae0d3655028951bf270ef7f4e7916a9424530951a89', 'preparacao': '9865042654c51e83955c2aeb186a1b5e27e1028a5c56f1f163e2891abc0bec49', 'divisao_10': 'a5dd35ff484dda77e8988aefdc961f05d07ea409d1336cbe90c4a5420632bab9'}


## 2. Carregar o código embutido

Todos os módulos necessários estão dentro deste notebook. A política V2 e
o associador histórico serão recuperados do ZIP autenticado do projeto.


In [ ]:
MODULOS_EMBUTIDOS = {'diag11_core.py': '"""Núcleo do diagnóstico 11. Não contém treinamento nem seleção de pesos."""\nfrom __future__ import annotations\n\nimport csv\nimport gzip\nimport hashlib\nimport io\nimport json\nimport math\nfrom collections import Counter, defaultdict\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nfrom PIL import Image\n\nNAMES = ["bed", "sofa", "chair", "table", "lamp", "tv", "laptop",\n         "wardrobe", "window", "door", "potted plant", "photo frame"]\nALL_KNOWN = (1 << len(NAMES)) - 1\nALERT_IDS = {0, 1, 2, 3, 7, 10}\nALERT_MASK = sum(1 << c for c in ALERT_IDS)\nLEVEL = {"sem_prioridade": 0, "baixo": 1, "medio": 2, "alto": 3}\n\n\ndef canonical(x):\n    return json.dumps(x, sort_keys=True, ensure_ascii=False, separators=(",", ":"))\n\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for block in iter(lambda: f.read(2**20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef json_clean(value):\n    if isinstance(value, dict):\n        return {str(k): json_clean(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [json_clean(v) for v in value]\n    if isinstance(value, np.generic):\n        return json_clean(value.item())\n    if isinstance(value, Path):\n        return str(value)\n    if isinstance(value, float) and not math.isfinite(value):\n        return None\n    return value\n\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    tmp.write_text(json.dumps(json_clean(value), ensure_ascii=False, indent=2,\n                              allow_nan=False), encoding="utf-8")\n    tmp.replace(path)\n\n\ndef save_csv(path, rows, fields=None):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    rows = list(rows)\n    if fields is None:\n        fields = list(dict.fromkeys(k for row in rows for k in row))\n    with path.open("w", newline="", encoding="utf-8-sig") as f:\n        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="raise")\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef safe_member(name):\n    p = PurePosixPath(name)\n    if p.is_absolute() or ".." in p.parts or "\\\\" in name:\n        raise ValueError(f"Membro inseguro no ZIP: {name}")\n    return p\n\n\ndef parse_yolo(raw, width, height, names=NAMES, source="label"):\n    objects = []\n    for line_no, line in enumerate(raw.decode("utf-8-sig").splitlines(), 1):\n        if not line.strip():\n            continue\n        parts = line.split()\n        if len(parts) != 5:\n            raise ValueError(f"{source}:{line_no}: esperados cinco campos.")\n        vals = [float(s) for s in parts]\n        if not all(math.isfinite(v) for v in vals):\n            raise ValueError(f"{source}:{line_no}: valor não finito.")\n        c, x, y, w, h = vals\n        if int(c) != c or not 0 <= c < len(names) or min(w, h) <= 0:\n            raise ValueError(f"{source}:{line_no}: classe ou dimensão inválida.")\n        box = [x-w/2, y-h/2, x+w/2, y+h/2]\n        if min(box) < -0.002 or max(box) > 1.002:\n            raise ValueError(f"{source}:{line_no}: caixa fora da imagem.")\n        box = [max(0, min(1, v)) for v in box]\n        objects.append({"classe_id": int(c), "classe": names[int(c)],\n                        "bbox": [box[0]*width, box[1]*height,\n                                 box[2]*width, box[3]*height],\n                        "ocluido": None, "truncado": None})\n    return objects\n\n\ndef hierarchy_mapping(cfg, class_rows, hierarchy):\n    if cfg["nomes"] != NAMES or len(cfg["mids_raizes"]) != len(NAMES):\n        raise ValueError("Classes da preparação incompatíveis com o projeto.")\n    descriptions = {r[0]: r[1] for r in class_rows if len(r) >= 2}\n    roots = cfg["mids_raizes"]\n    if any(mid not in descriptions for mid in roots):\n        raise ValueError("A descrição oficial não contém todas as raízes.")\n    descendants, ancestors = defaultdict(set), defaultdict(set)\n\n    def walk(node, parents=()):\n        mid = node["LabelName"]\n        ancestors[mid].update(parents)\n        descendants[mid].add(mid)\n        for parent in parents:\n            descendants[parent].add(mid)\n        for child in node.get("Subcategory", []):\n            walk(child, parents + (mid,))\n\n    walk(hierarchy)\n    excluded_names = set(cfg.get("ambiguas_excluidas", []))\n    excluded = {mid for mid, name in descriptions.items() if name in excluded_names}\n    candidates = defaultdict(set)\n    root_desc = {}\n    for cid, mid in enumerate(roots):\n        if mid not in descendants:\n            raise ValueError(f"Raiz ausente da hierarquia: {mid}")\n        root_desc[cid] = set(descendants[mid])\n        for sub in root_desc[cid]:\n            candidates[sub].add(cid)\n    ambiguous = excluded | {mid for mid, ids in candidates.items() if len(ids) > 1}\n    mapping = {mid: next(iter(ids)) for mid, ids in candidates.items()\n               if mid not in ambiguous}\n    return {"roots": roots, "descendants": root_desc, "ancestors": ancestors,\n            "mapping": mapping, "ambiguous": ambiguous,\n            "descriptions": descriptions,\n            "relevant_labels": set().union(*root_desc.values(),\n                                            *(ancestors[r] for r in roots))}\n\n\ndef prepare_oi_annotations(box_rows, label_rows, info):\n    """Cobertura conservadora: positivo na raiz ou negativo na raiz/ancestral.\n\n    Uma positiva apenas numa subclasse não prova cobertura exaustiva da classe\n    agregada. Grupos, representações, atributos desconhecidos ou caixas inválidas\n    tornam essa classe/imagem não avaliável, em vez de criarem falsos negativos.\n    """\n    labels = defaultdict(lambda: {"positive": set(), "negative": set()})\n    for row in label_rows:\n        if row.get("Source") not in {"verification", "crowdsource-verification"}:\n            continue\n        mid, confidence = row["LabelName"], float(row["Confidence"])\n        if mid not in info["relevant_labels"]:\n            continue\n        if confidence in (0, 1):\n            key = "positive" if confidence == 1 else "negative"\n            labels[row["ImageID"]][key].add(mid)\n    boxes, blocked = defaultdict(list), defaultdict(lambda: defaultdict(set))\n    audited = Counter()\n    for row in box_rows:\n        mid, iid = row["LabelName"], row["ImageID"]\n        affected = [cid for cid, mids in info["descendants"].items() if mid in mids]\n        if not affected:\n            continue\n        if mid in info["ambiguous"]:\n            for cid in affected:\n                blocked[iid][cid].add("subclasse_ambigua_excluida")\n            audited["caixas_subclasse_ambigua"] += 1\n            continue\n        cid = info["mapping"].get(mid)\n        if cid is None:\n            continue\n        flags = {k: int(float(row.get(k, -1))) for k in\n                 ["IsGroupOf", "IsDepiction", "IsOccluded", "IsTruncated"]}\n        reasons = []\n        if row.get("Source") not in {"xclick", "activemil"}:\n            reasons.append("origem_caixa_nao_reconhecida")\n        if flags["IsGroupOf"] != 0:\n            reasons.append("grupo_ou_atributo_desconhecido")\n        if flags["IsDepiction"] != 0:\n            reasons.append("representacao_ou_atributo_desconhecido")\n        xyxy = [float(row[k]) for k in ["XMin", "YMin", "XMax", "YMax"]]\n        if (not all(math.isfinite(v) for v in xyxy) or min(xyxy) < 0\n                or max(xyxy) > 1 or xyxy[2] <= xyxy[0] or xyxy[3] <= xyxy[1]):\n            reasons.append("coordenadas_invalidas")\n        if reasons:\n            blocked[iid][cid].update(reasons)\n            audited.update(reasons)\n            continue\n        boxes[iid].append({"classe_id": cid, "classe": NAMES[cid],\n                           "bbox_norm": xyxy, "mid_original": mid,\n                           "ocluido": flags["IsOccluded"] == 1 if flags["IsOccluded"] >= 0 else None,\n                           "truncado": flags["IsTruncated"] == 1 if flags["IsTruncated"] >= 0 else None})\n    records = {}\n    for iid in sorted(set(labels) | set(boxes)):\n        gt = []\n        seen = set()\n        for obj in boxes[iid]:\n            key = (obj["classe_id"], *(round(x, 8) for x in obj["bbox_norm"]))\n            if key not in seen:\n                seen.add(key)\n                gt.append(obj)\n        known, positive, negative, reasons = 0, 0, 0, {}\n        present = {o["classe_id"] for o in gt}\n        for cid, root in enumerate(info["roots"]):\n            pos = root in labels[iid]["positive"]\n            neg = bool(({root} | info["ancestors"][root]) & labels[iid]["negative"])\n            has_positive_label = bool(info["descendants"][cid] & labels[iid]["positive"])\n            reason = set(blocked[iid].get(cid, set()))\n            if neg and (cid in present or has_positive_label):\n                reason.add("contradicao_positivo_negativo")\n            if pos and cid not in present:\n                reason.add("positivo_sem_caixa_utilizavel")\n            if reason:\n                reasons[str(cid)] = sorted(reason)\n                continue\n            if neg:\n                known |= 1 << cid\n                negative |= 1 << cid\n            elif pos and cid in present:\n                known |= 1 << cid\n                positive |= 1 << cid\n        if positive or known & ALERT_MASK == ALERT_MASK:\n            records[iid] = {"id": iid, "source_split": "validation",\n                            "known_mask": known, "positive_mask": positive,\n                            "negative_mask": negative, "gt_normalizado": gt,\n                            "classes_positivas_avaliaveis": sorted(c for c in range(12)\n                                                                 if positive & (1 << c)),\n                            "restricoes_cobertura": reasons}\n    return records, dict(audited)\n\n\ndef iou_matrix(gt, pred):\n    if not gt or not pred:\n        return np.zeros((len(gt), len(pred)))\n    a = np.asarray([o["bbox"] for o in gt], dtype=float)\n    b = np.asarray([o["bbox"] for o in pred], dtype=float)\n    lt = np.maximum(a[:, None, :2], b[None, :, :2])\n    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])\n    intersection = np.maximum(rb-lt, 0).prod(axis=2)\n    aa = np.maximum(a[:, 2:]-a[:, :2], 0).prod(axis=1)\n    ba = np.maximum(b[:, 2:]-b[:, :2], 0).prod(axis=1)\n    return intersection / np.maximum(aa[:, None] + ba[None, :] - intersection, 1e-12)\n\n\ndef ap_matches(gt, pred, thresholds=np.linspace(.5, .95, 10)):\n    correct = np.zeros((len(pred), len(thresholds)), dtype=bool)\n    matrix = iou_matrix(gt, pred)\n    order = sorted(range(len(pred)), key=lambda pi: (-pred[pi]["confianca"], pi))\n    for ti, threshold in enumerate(thresholds):\n        used = set()\n        for pi in order:\n            options = [gi for gi, g in enumerate(gt) if gi not in used\n                       and g["classe_id"] == pred[pi]["classe_id"]\n                       and matrix[gi, pi] >= threshold]\n            if options:\n                gi = max(options, key=lambda k: (matrix[k, pi], -k))\n                used.add(gi)\n                correct[pi, ti] = True\n    return correct\n\n\ndef ratio(n, d):\n    return n/d if d else None\n\n\ndef alert_metrics(tp, fn, fp):\n    return {"tp": int(tp), "fn": int(fn), "fp": int(fp),\n            "precision": ratio(tp, tp+fp), "recall": ratio(tp, tp+fn),\n            "f1": ratio(2*tp, 2*tp+fp+fn)}\n\n\ndef alert_on(obj):\n    return bool(obj and LEVEL[obj["prioridade"]] >= LEVEL["medio"])\n\n\ndef with_policy(objects, width, height, policy):\n    return [{**o, **policy.avaliar_bbox(o["classe"], o["bbox"], width, height,\n                                       versao="v2")} for o in objects]\n\n\ndef record_event(record, model_name, event, g=None, p=None, iou=None, all_pred=()):\n    gbox = g["bbox"] if g else None\n    pbox = p["bbox"] if p else None\n    area = ((gbox[2]-gbox[0])*(gbox[3]-gbox[1]) /\n            (record["width"]*record["height"])) if gbox else None\n    side_at_640 = (min(gbox[2]-gbox[0], gbox[3]-gbox[1]) *\n                   640/max(record["width"], record["height"])) if gbox else None\n    flags = []\n    near_best = None\n    if event == "objeto_nao_detectado":\n        if side_at_640 < 32:\n            flags.append("objeto_pequeno_na_entrada_640")\n        if g.get("ocluido") is True:\n            flags.append("oclusao_anotada")\n        if g.get("truncado") is True:\n            flags.append("truncamento_anotado")\n        overlaps = iou_matrix([g], all_pred)[0] if all_pred else []\n        same = [i for i, o in enumerate(all_pred) if o["classe_id"] == g["classe_id"]]\n        if same:\n            best = max(same, key=lambda i: overlaps[i])\n            near_best = float(overlaps[best])\n            if overlaps[best] >= .5 and all_pred[best]["confianca"] < .25:\n                flags.append("deteccao_abaixo_da_confianca_025")\n            elif .1 <= overlaps[best] < .5:\n                flags.append("caixa_mesma_classe_iou_010_a_050")\n        if any(overlaps[i] >= .5 and o["classe_id"] != g["classe_id"]\n               and o["confianca"] >= .25 for i, o in enumerate(all_pred)):\n            flags.append("sobreposicao_com_previsao_de_outra_classe")\n        if not flags:\n            flags.append("omissao_sem_indicio_automatico")\n    if event == "previsao_nao_associada":\n        flags.append("previsao_sem_correspondencia_na_classe_avaliavel")\n    return {"coorte": record["coorte"], "imagem": record["name"], "id": record["id"],\n            "modelo": model_name, "evento": event,\n            "classe_id": g["classe_id"] if g else p["classe_id"],\n            "classe": g["classe"] if g else p["classe"],\n            "bbox_ref": json.dumps(gbox) if g else "",\n            "bbox_pred": json.dumps(pbox) if p else "",\n            "iou": iou, "confianca": p["confianca"] if p else None,\n            "prioridade_ref": g["prioridade"] if g else "sem_prioridade",\n            "prioridade_pred": p["prioridade"] if p else "sem_prioridade",\n            "ha_alerta_ref": alert_on(g), "ha_alerta_pred": alert_on(p),\n            "area_relativa_gt": area, "menor_lado_gt_em_640": side_at_640,\n            "oclusao_anotada": g.get("ocluido") if g else None,\n            "truncamento_anotado": g.get("truncado") if g else None,\n            "melhor_iou_mesma_classe": near_best,\n            "indicios": "|".join(flags)}\n\n\ndef analyse_prediction(record, pred_low, model_name, policy, matcher):\n    if record.get("role") not in {"home_val", "val_nova"}:\n        raise ValueError("Somente validação pode entrar na inferência do notebook 11.")\n    known = int(record["known_mask"])\n    gt = with_policy([g for g in record["gt"] if known & (1 << g["classe_id"])],\n                     record["width"], record["height"], policy)\n    pred_low_known = [p for p in pred_low if known & (1 << p["classe_id"])]\n    pred_all = with_policy([p for p in pred_low if p["confianca"] >= .25],\n                           record["width"], record["height"], policy)\n    pred = [p for p in pred_all if known & (1 << p["classe_id"])]\n    pairs, missing, extras = matcher(gt, pred)\n    events = []\n    for gi, pi, iou in pairs:\n        events.append(record_event(record, model_name, "correspondencia",\n                                   gt[gi], pred[pi], float(iou)))\n    for gi in missing:\n        events.append(record_event(record, model_name, "objeto_nao_detectado",\n                                   g=gt[gi], all_pred=pred_low_known))\n    for pi in extras:\n        events.append(record_event(record, model_name, "previsao_nao_associada", p=pred[pi]))\n    tp = sum(e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in events)\n    fn = sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in events)\n    fp = sum(not e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in events)\n    high_n = sum(g["prioridade"] == "alto" for g in gt)\n    high_missing = sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in events)\n    messages = [policy.gerar_mensagem(p, "medio") for p in pred_all if alert_on(p)]\n    if any(not isinstance(m, str) or not m.strip() for m in messages):\n        raise RuntimeError("A política produziu alerta sem texto.")\n    image = {"coorte": record["coorte"], "imagem": record["name"], "id": record["id"],\n             "modelo": model_name, "tp_alerta": tp, "fn_alerta": fn, "fp_alerta": fp,\n             "alta_ref": high_n, "alta_sem_alerta": high_missing,\n             "tp_detector": len(pairs), "fn_detector": len(missing), "fp_detector": len(extras),\n             "gt_avaliaveis": len(gt), "gt_ignorados": len(record["gt"])-len(gt),\n             "predicoes_ignoradas_conf_025": len(pred_all)-len(pred),\n             "cobertura_completa_alertas": known & ALERT_MASK == ALERT_MASK,\n             "ha_alerta_ref": any(alert_on(g) for g in gt),\n             "ha_alerta_pred": any(alert_on(p) for p in pred),\n             "mensagem_alerta": "\\n".join(messages)}\n    stats = (ap_matches(gt, pred_low_known),\n             np.array([p["confianca"] for p in pred_low_known], dtype=float),\n             np.array([p["classe_id"] for p in pred_low_known], dtype=int),\n             np.array([g["classe_id"] for g in gt], dtype=int))\n    return image, events, stats\n\n\ndef summarise(images, events, stats, ap_function=None):\n    if not images:\n        raise ValueError("Coorte de avaliação vazia.")\n    total = lambda k: sum(int(r[k]) for r in images)\n    output = {"imagens": len(images), "conf_alertas": .25,\n              "alertas_por_objeto": alert_metrics(total("tp_alerta"), total("fn_alerta"), total("fp_alerta")),\n              "alta_ref": total("alta_ref"), "alta_sem_alerta": total("alta_sem_alerta"),\n              "taxa_alta_sem_alerta": ratio(total("alta_sem_alerta"), total("alta_ref")),\n              "detector_conf_025": alert_metrics(total("tp_detector"), total("fn_detector"), total("fp_detector")),\n              "gt_ignorados_sem_cobertura": total("gt_ignorados"),\n              "predicoes_ignoradas_conf_025": total("predicoes_ignoradas_conf_025")}\n    complete = [r for r in images if r["cobertura_completa_alertas"]]\n    output["imagens_com_cobertura_completa_alertas"] = len(complete)\n    output["alertas_por_imagem_cobertura_completa"] = alert_metrics(\n        sum(r["ha_alerta_ref"] and r["ha_alerta_pred"] for r in complete),\n        sum(r["ha_alerta_ref"] and not r["ha_alerta_pred"] for r in complete),\n        sum(not r["ha_alerta_ref"] and r["ha_alerta_pred"] for r in complete))\n    correct = np.concatenate([s[0] for s in stats], axis=0)\n    scores = np.concatenate([s[1] for s in stats])\n    pred_cls = np.concatenate([s[2] for s in stats])\n    target_cls = np.concatenate([s[3] for s in stats])\n    ap = {}\n    if ap_function is not None and len(target_cls) and len(pred_cls):\n        result = ap_function(correct, scores, pred_cls, target_cls, plot=False,\n                             names=dict(enumerate(NAMES)))\n        values, ids = np.asarray(result[5]), np.asarray(result[6], dtype=int)\n        if values.shape != (len(ids), 10):\n            raise RuntimeError("Retorno de AP incompatível com a versão fixada.")\n        ap = {int(c): a for c, a in zip(ids, values)}\n    elif ap_function is not None and len(target_cls):\n        ap = {int(c): np.zeros(10) for c in np.unique(target_cls)}\n    output["map50_recorte"] = float(np.mean([a[0] for a in ap.values()])) if ap else None\n    output["map50_95_recorte"] = float(np.mean(list(ap.values()))) if ap else None\n    class_rows = []\n    for cid, name in enumerate(NAMES):\n        part = [e for e in events if e["classe_id"] == cid]\n        a = ap.get(cid)\n        n_gt = int((target_cls == cid).sum())\n        class_rows.append({"classe_id": cid, "classe": name, "gt_avaliaveis": n_gt,\n                           "ap50": float(a[0]) if a is not None else None,\n                           "ap50_95": float(a.mean()) if a is not None else None,\n                           "suporte_reduzido": n_gt < 30,\n                           **alert_metrics(sum(e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in part),\n                                           sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in part),\n                                           sum(not e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in part)),\n                           "alta_ref": sum(e["prioridade_ref"] == "alto" for e in part),\n                           "alta_sem_alerta": sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in part)})\n    return output, class_rows\n\n\ndef model_predictions(model, record, device):\n    if record["role"] not in {"home_val", "val_nova"}:\n        raise ValueError("Tentativa de inferência em reserva ou conjunto não autorizado.")\n    result = model.predict(str(record["image"]), conf=.001, imgsz=640, rect=True,\n                           iou=.70, max_det=300, device=device, half=False,\n                           verbose=False, save=False)[0]\n    if Path(result.path).resolve() != Path(record["image"]).resolve():\n        raise RuntimeError("A inferência retornou uma imagem diferente.")\n    if result.boxes is None:\n        return []\n    return [{"classe_id": int(c), "classe": NAMES[int(c)],\n             "bbox": [float(v) for v in box], "confianca": float(score)}\n            for c, box, score in zip(result.boxes.cls.cpu().numpy(),\n                                      result.boxes.xyxy.cpu().numpy(),\n                                      result.boxes.conf.cpu().numpy())]\n\n\ndef evaluate_model(model, records, model_name, output_dir, policy, matcher, device,\n                   ap_function):\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    images, events, stats = [], [], []\n    cache_file = output_dir / "predicoes_conf_0001.jsonl.gz"\n    with gzip.open(cache_file, "wt", encoding="utf-8") as cache:\n        for i, record in enumerate(records, 1):\n            pred = model_predictions(model, record, device)\n            cache.write(json.dumps({"id": record["id"], "imagem": record["name"],\n                                    "predicoes": pred}, ensure_ascii=False) + "\\n")\n            im, ev, st = analyse_prediction(record, pred, model_name, policy, matcher)\n            images.append(im)\n            events.extend(ev)\n            stats.append(st)\n            if i % 100 == 0 or i == len(records):\n                print(f"{record[\'coorte\']} / {model_name}: {i}/{len(records)}", flush=True)\n    summary, classes = summarise(images, events, stats, ap_function)\n    summary.update(modelo=model_name, coorte=records[0]["coorte"],\n                   protocolo="12 classes; AP por confiança; alertas por IoU; cobertura conservadora",\n                   nao_e_openimages_challenge=True, selecao_automatica_de_modelo=False)\n    save_json(output_dir / "metricas.json", summary)\n    save_csv(output_dir / "metricas_por_classe.csv", classes)\n    save_csv(output_dir / "alertas_por_imagem.csv", images)\n    save_csv(output_dir / "avaliacao_objetos.csv", events)\n    (output_dir / "alertas_texto.txt").write_text(\n        "\\n\\n".join(r["imagem"]+"\\n"+r["mensagem_alerta"] for r in images if r["mensagem_alerta"]),\n        encoding="utf-8")\n    return {"summary": summary, "classes": classes, "images": images, "events": events,\n            "cache_file": cache_file}\n', 'diag11_data.py': '"""Entradas, auditoria documental e nova validação, sem consulta a previsões."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport time\nimport urllib.request\nimport zipfile\nfrom collections import Counter, defaultdict\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nfrom scipy.fft import dctn\n\nfrom diag11_core import (ALL_KNOWN, ALERT_MASK, NAMES, canonical, parse_yolo,\n                         safe_member, save_csv, save_json, sha256_file)\n\nSOURCE_URLS = {\n    "classes": "https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions-boxable.csv",\n    "hierarquia": "https://storage.googleapis.com/openimages/2018_04/bbox_labels_600_hierarchy.json",\n    "caixas": "https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv",\n    "rotulos": "https://storage.googleapis.com/openimages/v5/validation-annotations-human-imagelabels-boxable.csv",\n    "metadata": "https://storage.googleapis.com/openimages/2018_04/validation/validation-images-with-rotation.csv",\n}\n\n\ndef download(url, path, retries=3):\n    """Download atômico, com recibo local e validação de tamanho quando informado."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    receipt = path.with_name(path.name + ".download.json")\n    if path.is_file() and receipt.is_file():\n        prior = json.loads(receipt.read_text())\n        if prior.get("url") == url and prior.get("sha256") == sha256_file(path):\n            return path\n    last = None\n    for attempt in range(retries):\n        tmp = path.with_name(path.name + ".tmp")\n        try:\n            req = urllib.request.Request(url, headers={"User-Agent": "TCC-Diagnostico-11/1.0"})\n            with urllib.request.urlopen(req, timeout=45) as response, tmp.open("wb") as out:\n                expected = response.headers.get("Content-Length")\n                for block in iter(lambda: response.read(2**20), b""):\n                    out.write(block)\n            if expected and tmp.stat().st_size != int(expected):\n                raise IOError("Tamanho recebido difere do Content-Length.")\n            if not tmp.stat().st_size:\n                raise IOError("Arquivo recebido vazio.")\n            tmp.replace(path)\n            save_json(receipt, {"url": url, "sha256": sha256_file(path),\n                                "bytes": path.stat().st_size})\n            return path\n        except Exception as exc:\n            last = type(exc).__name__ + ": " + str(exc)[:180]\n            tmp.unlink(missing_ok=True)\n            if attempt + 1 < retries:\n                time.sleep(min(2**attempt, 4))\n    raise IOError(f"Download não concluído: {url}; {last}")\n\n\ndef csv_rows(path):\n    with Path(path).open(encoding="utf-8-sig", newline="") as f:\n        yield from csv.DictReader(f)\n\n\ndef load_history(csv_path, json_path, expected_hash):\n    split = json.loads(Path(json_path).read_text())\n    stated = split.pop("divisao_sha256")\n    computed = hashlib.sha256(canonical(split).encode()).hexdigest()\n    if stated != expected_hash or computed != stated:\n        raise RuntimeError("Divisão histórica não corresponde à execução 10.")\n    rows = list(csv_rows(csv_path))\n    if len(rows) != 100000 or len({r["id"] for r in rows}) != 100000:\n        raise RuntimeError("O manifesto histórico precisa de 100.000 IDs distintos.")\n    if Counter(r["split"] for r in rows) != {"train": 80000, "test": 20000}:\n        raise RuntimeError("Divisão histórica diferente de 80/20.")\n    for label, key in [("train", "ids_treino"), ("test", "ids_teste")]:\n        if {r["id"] for r in rows if r["split"] == label} != set(split[key]):\n            raise RuntimeError("CSV e JSON históricos divergem.")\n    hashes = {r["imagem_sha256"] for r in rows}\n    if len(hashes) != len(rows):\n        raise RuntimeError("O histórico contém imagens repetidas em bytes.")\n    return {"ids": {r["id"] for r in rows}, "hashes": hashes, "rows": rows,\n            "split_hash": stated}\n\n\ndef phash(path):\n    with Image.open(path) as image:\n        small = np.asarray(image.convert("L").resize((32, 32), Image.Resampling.LANCZOS),\n                           dtype=float)\n    low = dctn(small, norm="ortho")[:8, :8]\n    bits = low > np.median(low.flatten()[1:])\n    bits[0, 0] = False\n    return int.from_bytes(np.packbits(bits.flatten()).tobytes(), "big")\n\n\nclass NearIndex:\n    """pHash de 64 bits, distância <=4; sinal de semelhança, não prova semântica."""\n    def __init__(self):\n        self.tables = [defaultdict(list) for _ in range(8)]\n\n    def add(self, value, name):\n        for band in range(8):\n            self.tables[band][(value >> (8*band)) & 255].append((value, name))\n\n    def find(self, value, max_distance=4):\n        checked = set()\n        for band in range(8):\n            for prior, name in self.tables[band].get((value >> (8*band)) & 255, []):\n                if (prior, name) in checked:\n                    continue\n                checked.add((prior, name))\n                if (prior ^ value).bit_count() <= max_distance:\n                    return name\n        return None\n\n\ndef extract_home(zip_path, destination, expected_zip, expected_test, expected_policy):\n    if sha256_file(zip_path) != expected_zip:\n        raise RuntimeError("ZIP Home diferente da entrada preservada.")\n    destination = Path(destination)\n    records, hashes, near = [], set(), NearIndex()\n    with zipfile.ZipFile(zip_path) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError("Há membros repetidos no ZIP Home.")\n        manifest = json.loads(z.read("verificacao_preparacao.json"))\n        if (manifest["test"]["sha256"] != expected_test or\n                manifest["test"]["imagens"] != 403 or\n                {int(k): v for k, v in manifest["classes"].items()} != dict(enumerate(NAMES))):\n            raise RuntimeError("Manifesto Home incompatível.")\n        root = manifest["dataset_dir"]\n        items = {r["arquivo"]: r for r in manifest["arquivos"]}\n        if len(items) != len(manifest["arquivos"]):\n            raise RuntimeError("Manifesto Home com membros repetidos.")\n        for name in z.namelist():\n            safe_member(name)\n            if name.startswith("codigo/") and name.endswith(".py"):\n                raw = z.read(name)\n                if name in items and hashlib.sha256(raw).hexdigest() != items[name]["sha256"]:\n                    raise RuntimeError("Código Home diferente do manifesto.")\n                out = destination / name\n                out.parent.mkdir(parents=True, exist_ok=True)\n                out.write_bytes(raw)\n        for name, item in sorted(items.items()):\n            is_image = (name.startswith(f"{root}/images/") and\n                        Path(name).suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"})\n            if is_image:\n                hashes.add(item["sha256"])\n            role = None\n            if name.startswith(f"{root}/images/train/raw_train/"):\n                role = "home_train"\n            elif name.startswith(f"{root}/images/val/raw_train/"):\n                role = "home_val"\n            if not is_image or role is None:\n                continue\n            raw = z.read(name)\n            if hashlib.sha256(raw).hexdigest() != item["sha256"]:\n                raise RuntimeError(f"Imagem Home alterada: {name}")\n            image = destination / name\n            image.parent.mkdir(parents=True, exist_ok=True)\n            image.write_bytes(raw)\n            label_name = name.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"\n            if label_name not in items:\n                raise RuntimeError("Label Home ausente do manifesto.")\n            label_raw = z.read(label_name)\n            if hashlib.sha256(label_raw).hexdigest() != items[label_name]["sha256"]:\n                raise RuntimeError("Label Home alterado.")\n            with Image.open(image) as im:\n                width, height = im.size\n                im.verify()\n            gt = parse_yolo(label_raw, width, height, source=label_name)\n            record = {"id": "home:" + name, "name": name.split("/raw_train/")[-1],\n                      "image": image, "role": role, "coorte": "home_val_historica",\n                      "width": width, "height": height, "known_mask": ALL_KNOWN,\n                      "sha256": item["sha256"], "label_sha256": items[label_name]["sha256"],\n                      "gt": gt}\n            record["phash"] = phash(image)\n            near.add(record["phash"], name)\n            records.append(record)\n    counts = Counter(r["role"] for r in records)\n    if counts != {"home_train": 1824, "home_val": 456}:\n        raise RuntimeError(f"Home inesperado: {dict(counts)}")\n    train_hash = {r["sha256"] for r in records if r["role"] == "home_train"}\n    val_hash = {r["sha256"] for r in records if r["role"] == "home_val"}\n    if train_hash & val_hash:\n        raise RuntimeError("Home TRAIN e VAL atuais compartilham imagens em bytes.")\n    policy_file = destination / "codigo/src/politica_alertas.py"\n    if not policy_file.is_file() or sha256_file(policy_file) != expected_policy:\n        raise RuntimeError("Política V2 diferente da versão preservada.")\n    audit = {"home_train_atual": 1824, "home_val_atual": 456,\n             "test_home_preservado": 403, "test_extraido": False,\n             "intersecao_bytes_home_train_val_atual": 0,\n             "divergencia_documental_1822_1824": "Não reconciliada; o pacote atual tem 1824.",\n             "separacao_atual_nao_comprova_historico_baseline": True}\n    return records, hashes, near, audit\n\n\ndef audit_baseline_metadata(checkpoint, expected_hash, home_records, evidence=None):\n    if sha256_file(checkpoint) != expected_hash:\n        raise RuntimeError("Peso baseline diferente da referência.")\n    # Só o checkpoint autenticado pelo SHA esperado é desserializado.\n    import torch\n    obj = torch.load(checkpoint, map_location="cpu", weights_only=False)\n    args = obj.get("train_args", {})\n    fields = ["data", "epochs", "batch", "imgsz", "seed", "optimizer", "lr0",\n              "project", "name", "fraction"]\n    audit = {"modelo_sha256": expected_hash, "data_checkpoint": str(obj.get("date")),\n             "train_args_preservados": {k: args[k] for k in fields if k in args},\n             "estado_proveniencia_original": "nao_determinado",\n             "home_val_usada_no_treino_original": None,\n             "motivo": "Caminhos e argumentos não contêm a lista de imagens vistas no treino original."}\n    del obj\n    if evidence is not None:\n        required = {"modelo_sha256", "train_image_sha256", "val_image_sha256"}\n        if not required <= evidence.keys() or evidence["modelo_sha256"] != expected_hash:\n            audit["evidencia_original"] = "incompativel_ou_sem_vinculo_ao_peso"\n        else:\n            train = set(evidence["train_image_sha256"])\n            val = set(evidence["val_image_sha256"])\n            valid_sha = lambda x: isinstance(x, str) and len(x) == 64 and all(c in "0123456789abcdef" for c in x)\n            if not train or any(not valid_sha(h) for h in train | val):\n                raise ValueError("Manifesto original precisa de hashes SHA-256 válidos.")\n            current = {r["sha256"] for r in home_records if r["role"] == "home_val"}\n            overlap = train & current\n            complete = evidence.get("manifesto_treino_completo") is True\n            audit.update(estado_proveniencia_original="manifesto_documental_analisado",\n                         evidencia_original="documento_vinculado_ao_sha_do_modelo",\n                         home_val_usada_no_treino_original=True if overlap else (False if complete else None),\n                         manifesto_declarado_completo=complete,\n                         imagens_home_val_no_manifesto_train_original=len(overlap),\n                         imagens_home_val_no_manifesto_val_original=len(current & val),\n                         motivo="Conclusão limitada à completude e correção do manifesto apresentado.")\n    return audit\n\n\ndef stable_order(iid, seed=42):\n    return hashlib.sha256(f"{seed}|{iid}".encode()).hexdigest(), iid\n\n\ndef balanced_ids(records, n, seed=42, per_class=45):\n    """Diversidade por classe e preenchimento determinístico, sem usar previsões."""\n    if len(records) < n:\n        raise ValueError(f"Há somente {len(records)} imagens elegíveis; exigidas {n}.")\n    groups = {c: sorted((iid for iid, r in records.items() if\n                         r["positive_mask"] & (1 << c)), key=lambda x: stable_order(x, seed))\n              for c in range(12)}\n    order = sorted(groups, key=lambda c: (len(groups[c]), c))\n    chosen, counts = set(), Counter()\n    # Diversificar exemplos negativos quando as seis classes têm ausência verificada.\n    negatives = sorted((iid for iid, r in records.items() if\n                        r["known_mask"] & ALERT_MASK == ALERT_MASK and\n                        not r["positive_mask"] & ALERT_MASK), key=lambda x: stable_order(x, seed))\n    for iid in negatives[:min(len(negatives), n//10)]:\n        chosen.add(iid)\n        counts.update(c for c in range(12) if records[iid]["positive_mask"] & (1 << c))\n    for target in range(1, per_class+1):\n        for cid in order:\n            if len(chosen) >= n:\n                break\n            if counts[cid] >= target:\n                continue\n            for iid in groups[cid]:\n                if iid not in chosen:\n                    chosen.add(iid)\n                    counts.update(c for c in range(12) if records[iid]["positive_mask"] & (1 << c))\n                    break\n    for iid in sorted(records, key=lambda x: stable_order(x, seed)):\n        if len(chosen) == n:\n            break\n        chosen.add(iid)\n    return sorted(chosen, key=lambda x: stable_order(x, seed))\n\n\ndef split_pool(records, n_reserve, seed=42):\n    """Estratos pela classe positiva mais rara; a reserva não entra na inferência."""\n    frequencies = Counter(c for r in records.values() for c in range(12)\n                          if r["positive_mask"] & (1 << c))\n    groups = defaultdict(list)\n    for iid, record in records.items():\n        classes = record["classes_positivas_avaliaveis"]\n        primary = min(classes, key=lambda c: (frequencies[c], c)) if classes else -1\n        groups[primary].append(iid)\n    total = len(records)\n    if not 0 < n_reserve < total:\n        raise ValueError("A reserva precisa ser menor que o conjunto novo.")\n    quotas = {c: len(ids)*n_reserve//total for c, ids in groups.items()}\n    order = sorted(groups, key=lambda c: (-(len(groups[c])*n_reserve % total), c))\n    for cid in order[:n_reserve-sum(quotas.values())]:\n        quotas[cid] += 1\n    selected = set()\n    for cid, ids in groups.items():\n        selected.update(sorted(ids, key=lambda x: stable_order(x, seed+1))[:quotas[cid]])\n    if len(selected) != n_reserve:\n        raise RuntimeError("A divisão não produziu a reserva planejada.")\n    return {iid: ("reserva_futura" if iid in selected else "val_nova") for iid in records}\n\n\ndef filter_metadata(records, metadata_path, prior_ids):\n    metadata, exclusions = {}, Counter()\n    for row in csv_rows(metadata_path):\n        iid = row["ImageID"]\n        if iid not in records:\n            continue\n        if row["Subset"] != "validation":\n            raise ValueError("A metadata contém uma origem diferente de validation.")\n        if iid in prior_ids:\n            exclusions["id_presente_nos_100k"] += 1\n            continue\n        try:\n            rotation = float(row["Rotation"])\n        except (TypeError, ValueError):\n            rotation = float("nan")\n        # Evita suposições sobre transformação das caixas e orientação do JPEG.\n        if rotation != 0:\n            exclusions["rotacao_nao_zero_ou_desconhecida"] += 1\n            continue\n        metadata[iid] = {k: row.get(k, "") for k in\n                         ["ImageID", "OriginalLandingURL", "License", "Author", "Rotation"]}\n    result = {iid: {**r, "creditos": metadata[iid]} for iid, r in records.items()\n              if iid in metadata}\n    return result, dict(exclusions)\n\n\ndef download_images(candidates, ids, directory, workers=8):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n\n    def one(iid):\n        record = dict(candidates[iid])\n        path = directory / f"oi_val_{iid}.jpg"\n        try:\n            download(f"https://open-images-dataset.s3.amazonaws.com/validation/{iid}.jpg", path)\n            with Image.open(path) as image:\n                image.verify()\n            with Image.open(path) as image:\n                width, height = image.size\n                orientation = image.getexif().get(274, 1)\n                if width < 32 or height < 32 or orientation != 1:\n                    raise ValueError("JPEG pequeno ou com orientação EXIF não trivial.")\n                image.load()\n            tiny = {g["classe_id"] for g in record["gt_normalizado"] if\n                    (g["bbox_norm"][2]-g["bbox_norm"][0])*width < 1 or\n                    (g["bbox_norm"][3]-g["bbox_norm"][1])*height < 1}\n            # A avaliação da ausência fica desconhecida para a classe afetada.\n            record["restricoes_cobertura"] = dict(record["restricoes_cobertura"])\n            for cid in tiny:\n                for key in ["known_mask", "positive_mask", "negative_mask"]:\n                    record[key] &= ~(1 << cid)\n                record["restricoes_cobertura"][str(cid)] = ["caixa_menor_que_um_pixel"]\n            record["classes_positivas_avaliaveis"] = [c for c in range(12)\n                                                      if record["positive_mask"] & (1 << c)]\n            if not (record["positive_mask"] or record["known_mask"] & ALERT_MASK == ALERT_MASK):\n                raise ValueError("Nenhuma cobertura útil após conferir as dimensões reais.")\n            record.update(image=path, width=width, height=height, sha256=sha256_file(path),\n                          phash=phash(path), name=path.name)\n            return record, None\n        except Exception as exc:\n            return None, {"id": iid, "motivo": type(exc).__name__, "detalhe": str(exc)[:240]}\n\n    valid, failures = {}, []\n    with ThreadPoolExecutor(max_workers=workers) as executor:\n        for done, (record, failure) in enumerate(executor.map(one, ids), 1):\n            if record:\n                valid[record["id"]] = record\n            else:\n                failures.append(failure)\n            if done % 100 == 0 or done == len(ids):\n                print(f"Novas imagens: {done}/{len(ids)}; válidas: {len(valid)}", flush=True)\n    return valid, failures\n\n\ndef deduplicate(records, prior_hashes, home_near, seed=42):\n    accepted, excluded, seen_hash, near = {}, [], set(), NearIndex()\n    for iid in sorted(records, key=lambda x: stable_order(x, seed)):\n        record = records[iid]\n        reason, similar = None, None\n        if record["sha256"] in prior_hashes:\n            reason = "bytes_presentes_nos_dados_anteriores"\n        elif record["sha256"] in seen_hash:\n            reason = "bytes_repetidos_no_conjunto_novo"\n        elif (similar := home_near.find(record["phash"])) is not None:\n            reason = "semelhanca_phash_com_home_train_ou_val"\n        elif (similar := near.find(record["phash"])) is not None:\n            reason = "semelhanca_phash_dentro_do_conjunto_novo"\n        if reason:\n            excluded.append({"id": iid, "motivo": reason, "semelhante_a": similar})\n            continue\n        accepted[iid] = record\n        seen_hash.add(record["sha256"])\n        near.add(record["phash"], iid)\n    return accepted, excluded\n\n\ndef freeze_dataset(records, roles, directory, source_hashes, prior_split_hash, seed=42):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    manifest, runtime = [], []\n    for iid in sorted(roles):\n        r, role = records[iid], roles[iid]\n        boxes = []\n        for g in r["gt_normalizado"]:\n            x1, y1, x2, y2 = g["bbox_norm"]\n            boxes.append({**g, "bbox": [x1*r["width"], y1*r["height"],\n                                       x2*r["width"], y2*r["height"]]})\n        coverage = {k: r[k] for k in ["known_mask", "positive_mask", "negative_mask",\n                                      "restricoes_cobertura"]}\n        label = directory / "labels" / role / f"oi_val_{iid}.txt"\n        coverage_file = directory / "coverage" / role / f"oi_val_{iid}.json"\n        label.parent.mkdir(parents=True, exist_ok=True)\n        lines = []\n        for g in r["gt_normalizado"]:\n            x1, y1, x2, y2 = g["bbox_norm"]\n            lines.append(f"{g[\'classe_id\']} {(x1+x2)/2:.10f} {(y1+y2)/2:.10f} {(x2-x1):.10f} {(y2-y1):.10f}")\n        label.write_text("\\n".join(lines)+("\\n" if lines else ""), encoding="utf-8")\n        save_json(coverage_file, coverage)\n        image_rel = f"images/{role}/oi_val_{iid}.jpg"\n        item = {"id": iid, "role": role, "image": image_rel,\n                "label": label.relative_to(directory).as_posix(),\n                "coverage": coverage_file.relative_to(directory).as_posix(),\n                "sha256": r["sha256"], "label_sha256": sha256_file(label),\n                "coverage_sha256": sha256_file(coverage_file),\n                "width": r["width"], "height": r["height"], "known_mask": r["known_mask"],\n                "positive_mask": r["positive_mask"], "negative_mask": r["negative_mask"],\n                "phash": f"{r[\'phash\']:016x}", "creditos": r["creditos"],\n                "gt_normalizado": r["gt_normalizado"]}\n        manifest.append(item)\n        runtime.append({**r, "role": role, "coorte": "openimages_val_nova",\n                        "gt": boxes, "label": label, "coverage": coverage_file})\n    metadata = {"versao": "11-validacao-independente-v1", "seed": seed, "nomes": NAMES,\n                "fontes_metadata_sha256": source_hashes, "origem_split": "validation",\n                "divisao_100k_anterior_sha256": prior_split_hash,\n                "contagens": dict(Counter(roles.values())),\n                "criterio": "Diversidade de classes, hash determinístico e pHash; sem previsões",\n                "cobertura": "Positivo explícito na raiz ou negativo na raiz/ancestral; classes problemáticas excluídas",\n                "independencia_limitada_ao_historico_conhecido": True,\n                "pretreinamentos_publicos_nao_auditados": True,\n                "reserva_futura_avaliada": False, "arquivos": manifest}\n    dataset_hash = hashlib.sha256(canonical(metadata).encode()).hexdigest()\n    metadata["dataset_sha256"] = dataset_hash\n    save_json(directory / "manifesto_validacao_nova.json", metadata)\n    save_csv(directory / "divisao.csv", [{k: r[k] for k in\n                                        ["id", "role", "sha256", "known_mask", "positive_mask"]}\n                                       for r in manifest])\n    archive = directory.parent / f"validacao_independente_11_{dataset_hash[:12]}.zip"\n    # Os JPEGs permanecem sem recompressão.\n    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_STORED) as z:\n        z.write(directory / "manifesto_validacao_nova.json", "manifesto_validacao_nova.json")\n        z.write(directory / "divisao.csv", "divisao.csv")\n        for r, item in zip(sorted(runtime, key=lambda r: r["id"]), manifest):\n            z.write(r["image"], item["image"])\n            z.write(r["label"], item["label"])\n            z.write(r["coverage"], item["coverage"])\n    return runtime, metadata, archive\n\n\ndef support_rows(records):\n    rows = []\n    for role in sorted({r["role"] for r in records}):\n        group = [r for r in records if r["role"] == role]\n        for cid, name in enumerate(NAMES):\n            gt = [g for r in group for g in r["gt"] if\n                  g["classe_id"] == cid and r["known_mask"] & (1 << cid)]\n            rows.append({"conjunto": role, "classe_id": cid, "classe": name,\n                         "imagens_cobertura_conhecida": sum(bool(r["known_mask"] & (1 << cid)) for r in group),\n                         "imagens_positivas_avaliaveis": sum(bool(r["positive_mask"] & (1 << cid)) for r in group),\n                         "caixas_avaliaveis": len(gt), "suporte_reduzido": len(gt) < 30})\n    return rows\n', 'diag11_reports.py': '"""Comparação pareada, indícios dos erros e galeria de inspeção."""\nfrom __future__ import annotations\n\nimport gzip\nimport html\nimport json\nimport zipfile\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont\n\nfrom diag11_core import (NAMES, LEVEL, alert_on, canonical, ratio, save_csv,\n                         save_json, with_policy)\nfrom diag11_data import stable_order\n\n\ndef paired_bootstrap(old_images, new_images, repetitions=1000, seed=42):\n    old = {r["id"]: r for r in old_images}\n    new = {r["id"]: r for r in new_images}\n    if set(old) != set(new) or len(old) != len(old_images) or len(new) != len(new_images):\n        raise ValueError("Bootstrap exige as mesmas imagens, sem repetições.")\n    ids = sorted(old)\n    fields = ["tp_alerta", "fn_alerta", "fp_alerta", "alta_sem_alerta", "alta_ref"]\n    a = np.array([[old[i][k] for k in fields] for i in ids], dtype=float)\n    b = np.array([[new[i][k] for k in fields] for i in ids], dtype=float)\n\n    def scores(x):\n        tp, fn, fp, missing, high = x\n        return [100*ratio(tp, tp+fp) if tp+fp else np.nan,\n                100*ratio(tp, tp+fn) if tp+fn else np.nan,\n                100*ratio(2*tp, 2*tp+fp+fn) if 2*tp+fp+fn else np.nan,\n                fn, fp, missing, 100*ratio(missing, high) if high else np.nan]\n\n    point = np.array(scores(b.sum(axis=0))) - np.array(scores(a.sum(axis=0)))\n    rng = np.random.default_rng(seed)\n    results = []\n    for _ in range(repetitions):\n        index = rng.integers(0, len(ids), len(ids))\n        results.append(np.array(scores(b[index].sum(axis=0))) -\n                       np.array(scores(a[index].sum(axis=0))))\n    samples = np.asarray(results)\n    labels = ["precision_alertas_pp", "recall_alertas_pp", "f1_alertas_pp",\n              "fn_alertas", "fp_alertas", "alta_sem_alerta", "taxa_alta_sem_alerta_pp"]\n    rows = []\n    for j, name in enumerate(labels):\n        values = samples[np.isfinite(samples[:, j]), j]\n        low, high = np.quantile(values, [.025, .975]) if len(values) else (None, None)\n        rows.append({"metrica": name, "delta_candidato_menos_antigo": float(point[j]) if np.isfinite(point[j]) else None,\n                     "ic95_inferior": low, "ic95_superior": high,\n                     "reamostragens_validas": len(values), "imagens": len(ids),\n                     "metodo": "Bootstrap pareado por imagem; não corrige possível exposição prévia"})\n    return rows\n\n\ndef comparison_rows(evaluations):\n    rows = []\n    for cohort, models in evaluations.items():\n        for name, result in models.items():\n            s, a = result["summary"], result["summary"]["alertas_por_objeto"]\n            rows.append({"coorte": cohort, "modelo": name, "imagens": s["imagens"],\n                         "precision_alertas": a["precision"], "recall_alertas": a["recall"],\n                         "f1_alertas": a["f1"], "tp_alerta": a["tp"], "fn_alerta": a["fn"],\n                         "fp_alerta": a["fp"], "alta_ref": s["alta_ref"],\n                         "alta_sem_alerta": s["alta_sem_alerta"],\n                         "map50_recorte": s["map50_recorte"],\n                         "map50_95_recorte": s["map50_95_recorte"]})\n    return rows\n\n\ndef diagnosis_rows(evaluations):\n    rows = []\n    for cohort, models in evaluations.items():\n        for model, result in models.items():\n            events = result["events"]\n            for cid, name in enumerate(NAMES):\n                group = [e for e in events if e["classe_id"] == cid]\n                for indication in sorted({x for e in group for x in e["indicios"].split("|") if x}):\n                    part = [e for e in group if indication in e["indicios"].split("|")]\n                    rows.append({"coorte": cohort, "modelo": model, "classe": name,\n                                 "indicio": indication, "eventos": len(part),\n                                 "fn_alerta": sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in part),\n                                 "alta_sem_alerta": sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in part),\n                                 "observacao": "Indícios podem se sobrepor; não estabelecem a causa do erro."})\n    return rows\n\n\ndef load_prediction_cache(path):\n    records = {}\n    with gzip.open(path, "rt", encoding="utf-8") as f:\n        for line in f:\n            r = json.loads(line)\n            if r["id"] in records:\n                raise ValueError("Cache tem imagem repetida.")\n            records[r["id"]] = r["predicoes"]\n    return records\n\n\ndef select_gallery_cases(old, new, limit=24, seed=42):\n    a, b = {r["id"]: r for r in old}, {r["id"]: r for r in new}\n    if set(a) != set(b):\n        raise ValueError("Galeria exige os dois modelos nas mesmas imagens.")\n    chosen = {}\n\n    def take(label, ordered, number):\n        used = 0\n        for iid in ordered:\n            if iid not in chosen and len(chosen) < limit:\n                chosen[iid] = label\n                used += 1\n                if used == number:\n                    break\n\n    worse = sorted(a, key=lambda i: (-(b[i]["alta_sem_alerta"]-a[i]["alta_sem_alerta"]),\n                                     -(b[i]["fn_alerta"]-a[i]["fn_alerta"]), stable_order(i, seed)))\n    better = sorted(a, key=lambda i: (b[i]["alta_sem_alerta"]-a[i]["alta_sem_alerta"],\n                                      b[i]["fn_alerta"]-a[i]["fn_alerta"], stable_order(i, seed)))\n    false = sorted(a, key=lambda i: (-(a[i]["fp_alerta"]+b[i]["fp_alerta"]), stable_order(i, seed)))\n    take("Candidato com mais omissões", [i for i in worse if\n         (b[i]["alta_sem_alerta"], b[i]["fn_alerta"]) >\n         (a[i]["alta_sem_alerta"], a[i]["fn_alerta"])], limit//4)\n    take("Candidato com menos omissões", [i for i in better if\n         (b[i]["alta_sem_alerta"], b[i]["fn_alerta"]) <\n         (a[i]["alta_sem_alerta"], a[i]["fn_alerta"])], limit//4)\n    take("Falsos alertas avaliáveis", [i for i in false if a[i]["fp_alerta"]+b[i]["fp_alerta"]], limit//4)\n    take("Amostra por hash, independente do erro", sorted(a, key=lambda i: stable_order(i, seed)), limit)\n    return [{"id": i, "categoria": label} for i, label in chosen.items()]\n\n\ndef font(size):\n    candidates = ["/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",\n                  "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf"]\n    for path in candidates:\n        if Path(path).is_file():\n            return ImageFont.truetype(path, size)\n    return ImageFont.load_default()\n\n\ndef draw_panel(record, objects, title):\n    canvas = Image.new("RGB", (540, 470), "#101a28")\n    draw = ImageDraw.Draw(canvas)\n    draw.text((12, 10), title, fill="#f5f7fa", font=font(18))\n    with Image.open(record["image"]) as image:\n        image = image.convert("RGB")\n        scale = min(520/image.width, 390/image.height)\n        image = image.resize((max(1, round(image.width*scale)),\n                              max(1, round(image.height*scale))), Image.Resampling.LANCZOS)\n    x0, y0 = (540-image.width)//2, 46+(390-image.height)//2\n    canvas.paste(image, (x0, y0))\n    sx, sy = image.width/record["width"], image.height/record["height"]\n    colours = {"alto": "#ff7b8a", "medio": "#ffde70",\n               "baixo": "#6ee7b7", "sem_prioridade": "#adbac7"}\n    for obj in objects:\n        cid = obj["classe_id"]\n        known = bool(record["known_mask"] & (1 << cid))\n        colour = colours[obj["prioridade"]] if known else "#a9afba"\n        x1, y1, x2, y2 = obj["bbox"]\n        box = [x0+x1*sx, y0+y1*sy, x0+x2*sx, y0+y2*sy]\n        draw.rectangle(box, outline=colour, width=2)\n        label = obj["classe"]\n        if "confianca" in obj:\n            label += f" {obj[\'confianca\']:.2f}"\n        label += " / " + obj["prioridade"] if known else " / sem cobertura"\n        # Limita o rótulo à largura visível; não altera a caixa nem as métricas.\n        tx, ty = max(0, min(box[0], 535)), max(44, box[1]-15)\n        text_box = draw.textbbox((tx, ty), label, font=font(11))\n        draw.rectangle(text_box, fill="#101a28")\n        draw.text((tx, ty), label, fill=colour, font=font(11))\n    draw.text((12, 445), "Rosa: alta | amarelo: média | verde: baixa | cinza: sem referência",\n              fill="#c6ced9", font=font(10))\n    return canvas\n\n\ndef render_case(record, old_pred, new_pred, policy, destination):\n    gt = with_policy([g for g in record["gt"] if record["known_mask"] & (1 << g["classe_id"])],\n                     record["width"], record["height"], policy)\n    old = with_policy([p for p in old_pred if p["confianca"] >= .25],\n                      record["width"], record["height"], policy)\n    new = with_policy([p for p in new_pred if p["confianca"] >= .25],\n                      record["width"], record["height"], policy)\n    panels = [draw_panel(record, gt, "Referência V2 avaliável"),\n              draw_panel(record, old, "Modelo antigo / confiança 0,25"),\n              draw_panel(record, new, "Candidato / confiança 0,25")]\n    result = Image.new("RGB", (1620, 470), "#101a28")\n    for i, panel in enumerate(panels):\n        result.paste(panel, (540*i, 0))\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    result.save(destination, quality=88)\n    return destination\n\n\ndef format_percent(value):\n    return "sem suporte" if value is None else f"{100*value:.2f}%"\n\n\ndef build_report(output_dir, evaluations, cohorts, audit, bootstrap, policy, dataset_manifest, seed=42):\n    output_dir = Path(output_dir)\n    comparison = comparison_rows(evaluations)\n    diagnosis = diagnosis_rows(evaluations)\n    save_csv(output_dir / "comparacao_modelos.csv", comparison)\n    save_csv(output_dir / "diagnostico_erros.csv", diagnosis)\n    save_csv(output_dir / "bootstrap_pareado.csv", bootstrap)\n    gallery = []\n    for cohort, models in evaluations.items():\n        records = {r["id"]: r for r in cohorts[cohort]}\n        caches = {name: load_prediction_cache(result["cache_file"]) for name, result in models.items()}\n        cases = select_gallery_cases(models["antigo"]["images"], models["candidato"]["images"], seed=seed)\n        old_images = {r["id"]: r for r in models["antigo"]["images"]}\n        new_images = {r["id"]: r for r in models["candidato"]["images"]}\n        for i, case in enumerate(cases):\n            record = records[case["id"]]\n            relative = f"galeria/{cohort}_{i:03}.jpg"\n            render_case(record, caches["antigo"][record["id"]],\n                        caches["candidato"][record["id"]], policy, output_dir/relative)\n            gallery.append({**case, "coorte": cohort, "imagem": record["name"], "arquivo": relative,\n                            "antigo": {k: old_images[record["id"]][k] for k in\n                                       ["tp_alerta", "fn_alerta", "fp_alerta", "alta_sem_alerta"]},\n                            "candidato": {k: new_images[record["id"]][k] for k in\n                                          ["tp_alerta", "fn_alerta", "fp_alerta", "alta_sem_alerta"]},\n                            "creditos": record.get("creditos", {})})\n    save_json(output_dir / "indice_galeria.json", gallery)\n    new_records = cohorts["openimages_val_nova"]\n    save_csv(output_dir / "revisao_cenarios.csv", [\n        {"id": r["id"], "imagem": r["name"], "ambiente_interno": "nao_revisado",\n         "proximo_do_uso_pretendido": "nao_revisado", "anotacao_parece_correta": "nao_revisado",\n         "observacao": "", "origem": r.get("creditos", {}).get("OriginalLandingURL", "")}\n        for r in new_records])\n    provenance = audit["baseline_original"]["estado_proveniencia_original"]\n    table = "<table><thead><tr><th>Conjunto</th><th>Modelo</th><th>Precisão alertas</th><th>Recall alertas</th><th>F1</th><th>FN</th><th>Alta sem alerta</th></tr></thead><tbody>"\n    for r in comparison:\n        table += "<tr>" + "".join("<td>"+html.escape(str(x))+"</td>" for x in\n                     [r["coorte"], r["modelo"], format_percent(r["precision_alertas"]),\n                      format_percent(r["recall_alertas"]), format_percent(r["f1_alertas"]),\n                      r["fn_alerta"], r["alta_sem_alerta"]]) + "</tr>"\n    table += "</tbody></table>"\n    cards = []\n    for case in gallery:\n        credit = case["creditos"]\n        source_url = str(credit.get("OriginalLandingURL", ""))\n        license_url = str(credit.get("License", ""))\n        attribution = html.escape(str(credit.get("Author", "")))\n        if source_url.startswith(("https://", "http://")):\n            attribution += \' · <a href="\'+html.escape(source_url, quote=True)+\'">Fonte original</a>\'\n        if license_url.startswith(("https://", "http://")):\n            attribution += \' · <a href="\'+html.escape(license_url, quote=True)+\'">Licença</a>\'\n        details = (f"Antigo: FN {case[\'antigo\'][\'fn_alerta\']}, FP {case[\'antigo\'][\'fp_alerta\']}, "\n                   f"alta sem alerta {case[\'antigo\'][\'alta_sem_alerta\']}. "\n                   f"Candidato: FN {case[\'candidato\'][\'fn_alerta\']}, FP {case[\'candidato\'][\'fp_alerta\']}, "\n                   f"alta sem alerta {case[\'candidato\'][\'alta_sem_alerta\']}.")\n        cards.append("<article><h3>"+html.escape(case["coorte"]+" / "+case["categoria"])+\n                     "</h3><p>"+html.escape(case["imagem"]+" — "+details)+"</p>"+\n                     \'<img loading="lazy" src="\'+case["arquivo"]+\'" alt="Caixas de referência e dos dois modelos">\'+\n                     "<p class=small>"+attribution+\n                     " · Sobreposição de caixas para avaliação.</p></article>")\n    html_report = """<!doctype html><html lang="pt-BR"><meta charset="utf-8">\n<meta name="viewport" content="width=device-width,initial-scale=1">\n<title>TCC — Diagnóstico dos detectores</title><style>\nbody{font:16px/1.6 system-ui,sans-serif;background:#f3f5f7;color:#142135;margin:0}\nmain{max-width:1400px;margin:auto;padding:32px}h1{font-size:32px;line-height:1.2}\n.note,article,table{background:white;border:1px solid #dbe1e8;border-radius:12px}\n.note,article{padding:20px;margin:20px 0}table{border-collapse:collapse;width:100%}\ntd,th{padding:10px;text-align:left;border-bottom:1px solid #e6eaf0}th{background:#e8eff8}\nimg{max-width:100%;height:auto}.small{font-size:12px;color:#526174}.scroll{overflow:auto}\n</style><main><h1>Diagnóstico comparativo dos detectores</h1>\n<p>Pesos congelados · 12 classes · política V2 · confiança 0,25 · sem treinamento</p>\n<div class=note><strong>Como interpretar</strong><p>\nA Home VAL é histórica e exploratória. A validação externa é nova em relação aos IDs\ne bytes dos dados conhecidos do projeto; pré-treinamentos públicos não foram auditados.\nO recorte Open Images tem cobertura parcial e não garante cenas internas.\nAs métricas de alertas medem concordância com a V2, não risco físico independente.\nA reserva futura não foi inferida. Nenhum modelo foi promovido automaticamente.</p>\n<p>Proveniência documental do baseline: <strong>""" + html.escape(provenance) + """</strong>.</p></div>\n<h2>Alertas por objeto avaliável</h2><div class=scroll>""" + table + """</div>\n<p>Os CSVs trazem suporte por classe, AP no recorte, contagens e intervalos pareados.\nNão compare APs de protocolos e distribuições diferentes como se fossem o mesmo benchmark.</p>\n<h2>Casos para inspeção</h2><p>Os casos abaixo priorizam diferenças e erros, além de uma amostra por hash.\nEsta galeria não é uma estimativa da frequência dos erros. As contagens acima usam todas as imagens.\nObjetos sem cobertura aparecem em cinza e não são classificados automaticamente como falsos positivos.</p>\n""" + "".join(cards) + """<h2>Próxima decisão</h2><p>\nRevisar os cenários e os indícios dos erros; só depois definir o treino de controle.\nOs pesos antigos, o candidato e os testes anteriores permanecem preservados.</p></main></html>"""\n    (output_dir / "relatorio_diagnostico.html").write_text(html_report, encoding="utf-8")\n    lines = ["# Diagnóstico 11", "", "Comparação sem novo treinamento e sem seleção automática.",\n             "", f"Proveniência original do baseline: {provenance}.",\n             f"Validação externa: {len(new_records)} imagens.",\n             f"Reserva futura não inferida: {dataset_manifest[\'contagens\'].get(\'reserva_futura\', 0)} imagens.",\n             "", "| Conjunto | Modelo | Precisão alertas | Recall alertas | F1 | FN | Alta sem alerta |",\n             "|---|---|---:|---:|---:|---:|---:|"]\n    for r in comparison:\n        lines.append("| "+" | ".join(map(str, [r["coorte"], r["modelo"],\n                     format_percent(r["precision_alertas"]), format_percent(r["recall_alertas"]),\n                     format_percent(r["f1_alertas"]), r["fn_alerta"], r["alta_sem_alerta"]]))+" |")\n    lines += ["", "A nova cobertura é conservadora e a seleção busca diversidade de classes; não representa a frequência natural do uso.",\n              "A Home VAL continua exploratória enquanto a origem do primeiro treinamento não estiver esclarecida.",\n              "Semelhança por pHash é um filtro de suspeitas, não uma prova de identidade de cena.",\n              "Os indícios de erro podem se sobrepor e não identificam causas de forma conclusiva.",\n              "A reserva atual não foi inferida; não há avaliação independente de risco físico.",\n              "", "Veja relatorio_diagnostico.html para a galeria e bootstrap_pareado.csv para a incerteza das diferenças."]\n    (output_dir / "relatorio_diagnostico.md").write_text("\\n".join(lines)+"\\n", encoding="utf-8")\n    return comparison, diagnosis, gallery\n\n\ndef package_reports(output_dir):\n    output_dir = Path(output_dir)\n    archive = output_dir / "pacote_diagnostico_11.zip"\n    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=4) as z:\n        for path in sorted(output_dir.rglob("*")):\n            if path.is_file() and path != archive and not path.name.endswith(".tmp"):\n                z.write(path, path.relative_to(output_dir))\n    return archive\n'}

PASTA_CODIGO = PASTA_LOCAL / "codigo_diagnostico"
PASTA_CODIGO.mkdir(exist_ok=True)
for nome, fonte in MODULOS_EMBUTIDOS.items():
    (PASTA_CODIGO / nome).write_text(fonte, encoding="utf-8")
sys.path.insert(0, str(PASTA_CODIGO))
import diag11_core as core
import diag11_data as dados
import diag11_reports as relatorios
NOMES_DICT = dict(enumerate(core.NAMES))


## 3. Conectar ao Drive e conferir entradas

A conexão é a mesma do notebook 10. A credencial fica em /tmp,
fora dos arquivos de saída. Os hashes autenticam os pesos e a base.


In [ ]:
from pathlib import Path
import base64
import configparser
import json
import shutil
import subprocess
import urllib.request
import zipfile
from kaggle_secrets import UserSecretsClient

SECRET_DRIVE = "GDRIVE_RCLONE_CONFIG_B64"
REMOTE_DRIVE = "gdrive_tcc"
ORIGEM_DRIVE = f"{REMOTE_DRIVE}:TCC_Visao_Computacional"
DRIVE_TCC = Path("/tmp/tcc_100k_entradas")
ENVIAR_RESULTADOS_AO_DRIVE = True
EXECUTAR_CALIBRACAO = False

# Credenciais ficam fora da pasta de resultados salva pelo Kaggle.
PASTA_RCLONE = Path("/tmp/tcc_rclone")
PASTA_RCLONE.mkdir(mode=0o700, parents=True, exist_ok=True)
PASTA_RCLONE.chmod(0o700)
CONFIG_RCLONE = PASTA_RCLONE / "gdrive_tcc.conf"
RCLONE = PASTA_RCLONE / "rclone"

try:
    segredo = UserSecretsClient().get_secret(SECRET_DRIVE)
except Exception:
    raise RuntimeError(
        f"Crie o Secret {SECRET_DRIVE} em Add-ons → Secrets e habilite "
        "o acesso deste notebook. Siga a preparação descrita acima."
    ) from None
try:
    configuracao_bytes = base64.b64decode(segredo.strip(), validate=True)
    configuracao = configparser.ConfigParser(interpolation=None)
    configuracao.read_string(configuracao_bytes.decode("utf-8-sig"))
except Exception:
    raise RuntimeError(
        "O Secret não contém a configuração rclone codificada corretamente. "
        "Copie novamente pelo comando PowerShell e atualize o Secret."
    ) from None
if (not configuracao.has_section(REMOTE_DRIVE)
        or configuracao.get(REMOTE_DRIVE, "type", fallback="") != "drive"
        or not configuracao.get(REMOTE_DRIVE, "token", fallback="")
        or not configuracao.get(REMOTE_DRIVE, "client_id", fallback="")):
    raise RuntimeError(
        "A configuração precisa da conexão gdrive_tcc, tipo drive, "
        "cliente OAuth próprio e autorização concluída no navegador."
    )
if (ENVIAR_RESULTADOS_AO_DRIVE
        and configuracao.get(REMOTE_DRIVE, "scope", fallback="") != "drive"):
    raise RuntimeError(
        "Para salvar os resultados, configure scope 1 (drive, leitura e "
        "gravação) no rclone e atualize o Secret."
    )
# Esta conexão usa Meu Drive, não um Drive compartilhado.
configuracao.remove_option(REMOTE_DRIVE, "team_drive")
with CONFIG_RCLONE.open("w", encoding="utf-8") as arquivo_config:
    configuracao.write(arquivo_config)
CONFIG_RCLONE.chmod(0o600)
del segredo, configuracao_bytes, configuracao

if not RCLONE.is_file():
    import os

    executavel = shutil.which("rclone")
    if executavel is None:
        prefixo = [] if os.geteuid() == 0 else ["sudo"]
        subprocess.run(prefixo + ["apt-get", "update", "-qq"], check=True)
        subprocess.run(
            prefixo + ["apt-get", "install", "-y", "--no-install-recommends", "rclone"],
            check=True,
        )
        executavel = shutil.which("rclone")

    if executavel is None:
        raise RuntimeError("A instalação terminou, mas o executável rclone não foi localizado.")

    shutil.copy2(executavel, RCLONE)
    RCLONE.chmod(0o700)
    print("rclone instalado pelo gerenciador de pacotes do Linux.")

DRIVE_TCC.mkdir(parents=True, exist_ok=True)
if shutil.disk_usage(DRIVE_TCC).free < 10 * 1024**3:
    raise RuntimeError("Reserve pelo menos 10 GiB livres para baixar as entradas.")

def baixar_arquivo_drive(relativo, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    print("Baixando:", relativo, flush=True)
    concluido = subprocess.run(
        [str(RCLONE), "--config", str(CONFIG_RCLONE), "copyto",
         f"{ORIGEM_DRIVE}/{relativo}", str(destino),
         "--retries", "3", "--low-level-retries", "5"], check=False,
    )
    if concluido.returncode != 0 or not destino.is_file():
        raise RuntimeError(f"Falha ao baixar {relativo}. Confira a saída do rclone, a autorização e a Internet.")
    return destino


def enviar_resultados_drive(etapa="manual", obrigatorio=False):
    if not ENVIAR_RESULTADOS_AO_DRIVE:
        return False
    pasta = Path(PASTA_RESULTADOS)
    destino = f"{ORIGEM_DRIVE}/resultados/{NOME_EXPERIMENTO}"
    print(f"Enviando resultados ao Drive ({etapa})...", flush=True)
    try:
        enviado = subprocess.run(
            [str(RCLONE), "--config", str(CONFIG_RCLONE), "copy",
             str(pasta), destino, "--exclude", "**/*.tmp",
             "--retries", "2", "--low-level-retries", "3",
             "--contimeout", "15s", "--timeout", "60s"],
            check=False,
        )
        if enviado.returncode != 0:
            raise RuntimeError(f"rclone retornou código {enviado.returncode}")
    except (OSError, RuntimeError) as exc:
        mensagem = (
            f"Envio ao Drive falhou em {etapa}: {exc}. "
            "Os arquivos locais continuam em /kaggle/working/resultados_tcc. "
            "Confira a autorização e execute a última etapa para repetir o envio."
        )
        if obrigatorio:
            raise RuntimeError(mensagem) from None
        print(mensagem, flush=True)
        return False
    print("Resultados copiados ao Drive.", flush=True)
    return True

def enviar_resultados(etapa):
    return enviar_resultados_drive(etapa, obrigatorio=True)

def comando_drive(args, descricao):
    result = subprocess.run([str(RCLONE), "--config", str(CONFIG_RCLONE)] + args,
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if result.returncode:
        raise RuntimeError("Falha ao " + descricao + ". Confira Internet e Secret.")
    return result.stdout

ENTRADAS = PASTA_LOCAL / "entradas"
ZIP_HOME = baixar_arquivo_drive("datasets/homeobjects_sunrgbd_validado.zip", ENTRADAS/"home.zip")
PREP = baixar_arquivo_drive("datasets/openimages_v7_tcc_100k_v1/configuracao_preparacao.json",
                           ENTRADAS/"configuracao_preparacao.json")
MODELOS = {
    "antigo": baixar_arquivo_drive("modelos/yolov8n_baseline/best.pt", ENTRADAS/"baseline.pt"),
    "candidato": baixar_arquivo_drive(PASTA_EXECUCAO_10+"/best.pt", ENTRADAS/"candidato_10.pt"),
}
DIVISAO_CSV = baixar_arquivo_drive(PASTA_EXECUCAO_10+"/divisao_openimages.csv", ENTRADAS/"divisao_10.csv")
DIVISAO_JSON = baixar_arquivo_drive(PASTA_EXECUCAO_10+"/divisao_openimages.json", ENTRADAS/"divisao_10.json")
CONFIG_10 = baixar_arquivo_drive(PASTA_EXECUCAO_10+"/configuracao_experimento.json", ENTRADAS/"configuracao_10.json")
for name, key in [("antigo", "baseline"), ("candidato", "candidato")]:
    if core.sha256_file(MODELOS[name]) != EXPECTED[key]:
        raise RuntimeError("Identidade inesperada do peso " + name)
cfg = json.loads(PREP.read_text())
if hashlib.sha256(core.canonical(cfg).encode()).hexdigest() != EXPECTED["preparacao"]:
    raise RuntimeError("O mapeamento da preparação foi alterado.")
config_10 = json.loads(CONFIG_10.read_text())
if (config_10.get("versao") != "10-openimages-80-20-1ep-3.0" or
        config_10.get("divisao_openimages_sha256") != EXPECTED["divisao_10"]):
    raise RuntimeError("Pasta diferente da execução 10 avaliada.")
HISTORICO = dados.load_history(DIVISAO_CSV, DIVISAO_JSON, EXPECTED["divisao_10"])
print("Pesos, classes e 100 mil IDs históricos conferidos.")


## 4. Auditar Home e a documentação do baseline

Conferir a separação atual de Home TRAIN/VAL não comprova o que o baseline
viu no treinamento original. Caminhos em train_args não provam essa ausência.

Se existir modelos/yolov8n_baseline/manifesto_treino_baseline.json, ele será
analisado. Sem evidência histórica, a origem fica **nao_determinado** e a
execução continua na validação nova. Nenhum manifesto retrospectivo é
criado como se fosse prova do treinamento.


In [ ]:
HOME, HASHES_HOME, PHASH_HOME, AUDITORIA_HOME = dados.extract_home(
    ZIP_HOME, PASTA_LOCAL/"home_extraido", EXPECTED["home_zip"],
    EXPECTED["test_home"], EXPECTED["politica"])
sys.path.insert(0, str(PASTA_LOCAL/"home_extraido/codigo"))
import src.politica_alertas as politica
import src.validacao_alertas as avaliacao_base
if core.sha256_file(politica.__file__) != EXPECTED["politica"]:
    raise RuntimeError("Política V2 diferente da preservada.")
if (float(avaliacao_base.IOU_CORRESPONDENCIA) != .5 or
        float(avaliacao_base.IOU_NMS) != .7 or int(avaliacao_base.IMGSZ) != 640):
    raise RuntimeError("Associador histórico com parâmetros diferentes.")
if "versao" not in inspect.signature(politica.avaliar_bbox).parameters:
    raise RuntimeError("Política sem o contrato V2 esperado.")
evidence = None
arquivos_baseline = json.loads(comando_drive(
    ["lsjson", f"{ORIGEM_DRIVE}/modelos/yolov8n_baseline",
     "--files-only", "--max-depth", "1"], "listar documentação do baseline"))
if any(r["Name"] == "manifesto_treino_baseline.json" for r in arquivos_baseline):
    p = baixar_arquivo_drive("modelos/yolov8n_baseline/manifesto_treino_baseline.json",
                             ENTRADAS/"manifesto_treino_baseline.json")
    evidence = json.loads(p.read_text())
AUDITORIA_BASELINE = dados.audit_baseline_metadata(
    MODELOS["antigo"], EXPECTED["baseline"], HOME, evidence)
AUDITORIA = {"separacao_atual": AUDITORIA_HOME, "baseline_original": AUDITORIA_BASELINE,
             "test_home_extraido_ou_inferido": False, "test_oi20k_inferido": False,
             "novos_treinamentos": 0}
core.save_json(PASTA_RESULTADOS/"auditoria_baseline.json", AUDITORIA)
print("Home atual: 1824 TRAIN / 456 VAL / 403 TEST preservadas.")
print("Proveniência original:", AUDITORIA_BASELINE["estado_proveniencia_original"])
print("Home VAL vista no treino original:", AUDITORIA_BASELINE["home_val_usada_no_treino_original"])
enviar_resultados("auditoria das entradas")


## 5. Selecionar novas imagens pelas anotações humanas

A origem é o split oficial **validation** do Open Images. As raízes e
exclusões são as mesmas da configuração autenticada da preparação.

A cobertura é conservadora: positivo explícito na raiz e com caixas úteis,
ou negativo na raiz/ancestral. Positivo somente numa subclasse não comprova
cobertura completa da classe agregada. Grupos, representações, coordenadas
inválidas e contradições tornam a classe/imagem não avaliável.
**Classes desconhecidas não viram negativos.**

A seleção busca diversidade por classe e usa ordenação determinística,
antes de qualquer previsão. As proporções não representam a frequência
natural de uso. Classes raras podem continuar com pouco suporte.


In [ ]:
FONTES = PASTA_LOCAL/"metadata_openimages"
FONTES.mkdir(exist_ok=True)
def buscar_fonte(item):
    key, url = item
    path = dados.download(url, FONTES/(key+".dat"))
    print("Metadata pronta:", key, flush=True)
    return key, path
with ThreadPoolExecutor(max_workers=5) as executor:
    ARQUIVOS_FONTES = dict(executor.map(buscar_fonte, dados.SOURCE_URLS.items()))
with ARQUIVOS_FONTES["classes"].open(encoding="utf-8-sig", newline="") as f:
    CLASSES_OFICIAIS = list(csv.reader(f))
MAPA = core.hierarchy_mapping(cfg, CLASSES_OFICIAIS,
                             json.loads(ARQUIVOS_FONTES["hierarquia"].read_text()))
CANDIDATOS_METADATA, FILTROS_CAIXAS = core.prepare_oi_annotations(
    dados.csv_rows(ARQUIVOS_FONTES["caixas"]), dados.csv_rows(ARQUIVOS_FONTES["rotulos"]), MAPA)
CANDIDATOS_METADATA, FILTROS_METADATA = dados.filter_metadata(
    CANDIDATOS_METADATA, ARQUIVOS_FONTES["metadata"], HISTORICO["ids"])
N_TOTAL = N_VALIDACAO + N_RESERVA_FUTURA
if len(CANDIDATOS_METADATA) < N_TOTAL:
    raise RuntimeError(f"Há {len(CANDIDATOS_METADATA)} elegíveis para {N_TOTAL} solicitadas; examine a cobertura.")
N_DOWNLOAD = min(len(CANDIDATOS_METADATA), N_TOTAL + SOBRESSALENTES_DOWNLOAD)
IDS_DOWNLOAD = dados.balanced_ids(CANDIDATOS_METADATA, N_DOWNLOAD, SEED)
HASHES_FONTES = {name: core.sha256_file(path) for name, path in ARQUIVOS_FONTES.items()}
INVENTARIO = [{"classe": name, "imagens_positivas_avaliaveis_disponiveis":
               sum(bool(r["positive_mask"] & (1 << cid)) for r in CANDIDATOS_METADATA.values())}
              for cid, name in NOMES_DICT.items()]
core.save_csv(PASTA_RESULTADOS/"inventario_externo_disponivel.csv", INVENTARIO)
core.save_json(PASTA_RESULTADOS/"fontes_e_filtros.json",
               {"urls": dados.SOURCE_URLS, "sha256": HASHES_FONTES,
                "filtros_caixas": FILTROS_CAIXAS, "filtros_metadata": FILTROS_METADATA,
                "elegiveis": len(CANDIDATOS_METADATA), "planejadas_download": N_DOWNLOAD})
core.save_json(PASTA_RESULTADOS/"plano_download_antes_inferencia.json",
               {"seed": SEED, "ids": IDS_DOWNLOAD, "modelos_ainda_nao_inferidos": True})
display(pd.DataFrame(INVENTARIO))
print("Download planejado, com sobressalentes:", N_DOWNLOAD)


## 6. Baixar, excluir repetições e congelar a divisão

Excluímos IDs/bytes antigos, repetições internas e suspeitas fortes de
semelhança por pHash. O pHash é conferido entre as novas imagens e contra
Home TRAIN/VAL; não temos pHashes de todo o histórico Open Images ou de
pré-treinamentos públicos. Semelhança visual não prova identidade de cena.

A orientação é conferida conservadoramente e os JPEGs CVDF ficam sem
recompressão. As 300 imagens de reserva não entram na inferência.


In [ ]:
IMAGENS_BAIXADAS, FALHAS_DOWNLOAD = dados.download_images(
    CANDIDATOS_METADATA, IDS_DOWNLOAD, PASTA_LOCAL/"novas_imagens", WORKERS_DOWNLOAD)
IMAGENS_LIMPAS, EXCLUSOES_REPETICAO = dados.deduplicate(
    IMAGENS_BAIXADAS, HISTORICO["hashes"] | HASHES_HOME, PHASH_HOME, SEED)
core.save_csv(PASTA_RESULTADOS/"falhas_download.csv", FALHAS_DOWNLOAD,
              fields=["id", "motivo", "detalhe"])
core.save_csv(PASTA_RESULTADOS/"exclusoes_repeticao.csv", EXCLUSOES_REPETICAO,
              fields=["id", "motivo", "semelhante_a"])
if len(IMAGENS_LIMPAS) < N_TOTAL:
    enviar_resultados("diagnóstico de imagens insuficientes")
    raise RuntimeError(f"Restaram {len(IMAGENS_LIMPAS)} imagens para {N_TOTAL} exigidas. Relatórios preservados; não reduza silenciosamente o conjunto.")
IDS_FINAIS = dados.balanced_ids(IMAGENS_LIMPAS, N_TOTAL, SEED)
POOL = {iid: IMAGENS_LIMPAS[iid] for iid in IDS_FINAIS}
PAPEIS = dados.split_pool(POOL, N_RESERVA_FUTURA, SEED)
NOVAS, MANIFESTO_NOVO, PACOTE_DADOS = dados.freeze_dataset(
    POOL, PAPEIS, PASTA_LOCAL/"dataset_novo", HASHES_FONTES, HISTORICO["split_hash"], SEED)
VAL_NOVA = [r for r in NOVAS if r["role"] == "val_nova"]
RESERVA_NOVA = [r for r in NOVAS if r["role"] == "reserva_futura"]
if len(VAL_NOVA) != N_VALIDACAO or len(RESERVA_NOVA) != N_RESERVA_FUTURA:
    raise RuntimeError("Divisão nova diferente do planejamento.")
if ({r["id"] for r in VAL_NOVA} & {r["id"] for r in RESERVA_NOVA} or
        {r["sha256"] for r in NOVAS} & (HISTORICO["hashes"] | HASHES_HOME)):
    raise RuntimeError("Sobreposição de identidade no conjunto novo.")
core.save_json(PASTA_RESULTADOS/"manifesto_validacao_nova.json", MANIFESTO_NOVO)
SUPORTE = dados.support_rows(NOVAS)
core.save_csv(PASTA_RESULTADOS/"suporte_por_classe.csv", SUPORTE)
display(pd.DataFrame(SUPORTE))
DATASET_SHA = MANIFESTO_NOVO["dataset_sha256"]
DESTINO_DATASET = f"datasets/validacao_independente_11_v1/{DATASET_SHA}"
comando_drive(["copyto", str(PACOTE_DADOS),
               f"{ORIGEM_DRIVE}/{DESTINO_DATASET}/{PACOTE_DADOS.name}"], "salvar recorte novo")
comando_drive(["copyto", str(PASTA_RESULTADOS/"manifesto_validacao_nova.json"),
               f"{ORIGEM_DRIVE}/{DESTINO_DATASET}/manifesto_validacao_nova.json"], "salvar manifesto novo")
core.save_json(PASTA_RESULTADOS/"localizacao_dataset.json",
               {"pasta_drive": DESTINO_DATASET, "arquivo": PACOTE_DADOS.name,
                "arquivo_sha256": core.sha256_file(PACOTE_DADOS), "dataset_sha256": DATASET_SHA,
                "validacao": N_VALIDACAO, "reserva_futura": N_RESERVA_FUTURA})
enviar_resultados("divisão nova congelada")
print("Validação congelada:", len(VAL_NOVA), "| reserva sem inferência:", len(RESERVA_NOVA))


## 7. Congelar o protocolo e comparar os pesos

Alertas em **0,25**; curva AP em **0,001**. A correspondência dos alertas
reutiliza o associador histórico por classe e IoU mínima 0,50.
AP usa correspondência por confiança no recorte de cobertura conhecida.

A inferência é por imagem, FP32, rect=True, resolução 640, NMS 0,70,
máximo de 300 detecções. O mesmo procedimento vale para os dois pesos.
As coortes são analisadas separadamente. Não calibramos nem escolhemos pesos.


In [ ]:
HOME_VAL = [r for r in HOME if r["role"] == "home_val"]
COORTES = {"home_val_historica": HOME_VAL, "openimages_val_nova": VAL_NOVA}
IDS_RESERVA = {r["id"] for r in RESERVA_NOVA}
if any(r["id"] in IDS_RESERVA or r["role"] not in {"home_val", "val_nova"}
       for cohort in COORTES.values() for r in cohort):
    raise RuntimeError("Conjunto não autorizado entrou na avaliação.")
HASHES_MODELOS_INICIAIS = {name: core.sha256_file(path) for name, path in MODELOS.items()}
CONFIGURACAO = {
    "versao": VERSAO, "experimento": NOME_EXPERIMENTO, "seed": SEED,
    "modelos_sha256": HASHES_MODELOS_INICIAIS, "dataset_sha256": DATASET_SHA,
    "politica_sha256": EXPECTED["politica"], "conf_alertas": .25, "conf_ap": .001,
    "iou_nms": .70, "iou_correspondencia": .50, "imgsz": 640,
    "max_det": 300, "rect": True, "half": False, "batch_inferencia": 1,
    "novos_treinamentos": 0, "limiares_calibrados": False,
    "selecao_automatica_de_modelo": False, "reserva_futura_inferida": False,
    "coortes": {name: len(records) for name, records in COORTES.items()},
    "codigo_embutido_sha256": {name: hashlib.sha256(source.encode()).hexdigest()
                              for name, source in MODULOS_EMBUTIDOS.items()},
    "ambiente": {"python": platform.python_version(), "torch": torch.__version__,
                 "ultralytics": ultralytics.__version__, "device": str(DEVICE)}}
core.save_json(PASTA_RESULTADOS/"configuracao_diagnostico.json", CONFIGURACAO)
enviar_resultados("protocolo antes da inferência")
AVALIACOES = {name: {} for name in COORTES}
for model_name, path in MODELOS.items():
    model = YOLO(str(path))
    if {int(k): str(v) for k, v in model.names.items()} != NOMES_DICT:
        raise RuntimeError("Mapa de classes incompatível em " + model_name)
    for cohort, records in COORTES.items():
        AVALIACOES[cohort][model_name] = core.evaluate_model(
            model, records, model_name, PASTA_RESULTADOS/cohort/model_name,
            politica, avaliacao_base.corresponder_objetos, DEVICE, ap_per_class)
        enviar_resultados(cohort + " / " + model_name)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print("Dois pesos comparados; reserva futura fora da inferência.")


## 8. Comparar diferenças e sua incerteza

Bootstrap pareado por imagem, com mil reamostragens. Intervalos que cruzam
zero não demonstram uma direção consistente da diferença nessa análise.
Isso não corrige possível exposição histórica nem dependência entre fotos
da mesma cena. O suporte por classe continua necessário.

As contagens históricas da Home são conferidas como teste de reprodução,
sem alterar parâmetros para forçar coincidência.


In [ ]:
BOOTSTRAP = []
for cohort, models in AVALIACOES.items():
    for row in relatorios.paired_bootstrap(models["antigo"]["images"],
                                           models["candidato"]["images"], 1000, SEED):
        BOOTSTRAP.append({"coorte": cohort, **row})
EXPECTATIVAS_HOME = {
    "antigo": {"tp": 1380, "fn": 155, "fp": 381, "alta_sem_alerta": 24},
    "candidato": {"tp": 1359, "fn": 176, "fp": 382, "alta_sem_alerta": 27}}
REPRODUCAO = {}
for name, expected in EXPECTATIVAS_HOME.items():
    summary = AVALIACOES["home_val_historica"][name]["summary"]
    observed = {**{k: summary["alertas_por_objeto"][k] for k in ["tp", "fn", "fp"]},
                "alta_sem_alerta": summary["alta_sem_alerta"]}
    REPRODUCAO[name] = {"esperado_execucao10": expected, "observado": observed,
                        "contagens_iguais": expected == observed}
core.save_json(PASTA_RESULTADOS/"reproducao_home_val.json", REPRODUCAO)
display(pd.DataFrame(relatorios.comparison_rows(AVALIACOES)))
display(pd.DataFrame(BOOTSTRAP))
print("Reprodução Home VAL:", {k: v["contagens_iguais"] for k, v in REPRODUCAO.items()})
if not all(v["contagens_iguais"] for v in REPRODUCAO.values()):
    print("Diferença de reprodução: revisar versões, inferência e associador antes de atribuir a mudança ao modelo.")


## 9. Produzir galeria e indícios dos erros

Cada exemplo mostra **referência / antigo / candidato**. A galeria contém
diferenças entre modelos, falsos alertas e uma amostra por hash. As métricas
usam todas as imagens; a galeria serve para inspeção.

Os indícios incluem tamanho pequeno na entrada 640, oclusão anotada,
truncamento, IoU insuficiente, previsão de outra classe e confiança abaixo
de 0,25. Eles podem se sobrepor e não provam a causa.

revisao_cenarios.csv permite registrar depois se as cenas são internas
e próximas do uso pretendido. O notebook conclui sem preenchimento manual.


In [ ]:
COMPARACAO, DIAGNOSTICOS, GALERIA = relatorios.build_report(
    PASTA_RESULTADOS, AVALIACOES, COORTES, AUDITORIA, BOOTSTRAP,
    politica, MANIFESTO_NOVO, SEED)
display(pd.DataFrame(COMPARACAO))
display(pd.DataFrame(DIAGNOSTICOS).head(20))
print("Casos visuais:", len(GALERIA))
display(FileLink(str(PASTA_RESULTADOS/"relatorio_diagnostico.html")))


## 10. Conferir integridade e salvar o pacote final

O arquivo de resultados é **pacote_diagnostico_11.zip**.
O conjunto novo reutilizável é salvo separadamente no Drive, com JPEGs,
manifesto, labels e cobertura. A reserva futura não foi inferida.


In [ ]:
for name, path in MODELOS.items():
    if core.sha256_file(path) != HASHES_MODELOS_INICIAIS[name]:
        raise RuntimeError("Peso alterado durante o diagnóstico.")
if core.sha256_file(politica.__file__) != EXPECTED["politica"]:
    raise RuntimeError("Política alterada durante o diagnóstico.")
for record in HOME + NOVAS:
    if core.sha256_file(record["image"]) != record["sha256"]:
        raise RuntimeError("Imagem alterada durante o diagnóstico.")
manifest_check = dict(MANIFESTO_NOVO)
claimed_hash = manifest_check.pop("dataset_sha256")
if hashlib.sha256(core.canonical(manifest_check).encode()).hexdigest() != claimed_hash:
    raise RuntimeError("Manifesto novo alterado.")
for record, item in zip(sorted(NOVAS, key=lambda r: r["id"]), MANIFESTO_NOVO["arquivos"]):
    if (core.sha256_file(record["label"]) != item["label_sha256"] or
            core.sha256_file(record["coverage"]) != item["coverage_sha256"]):
        raise RuntimeError("Rótulo ou cobertura nova alterados.")
RESUMO = {
    "versao": VERSAO, "experimento": NOME_EXPERIMENTO, "concluido": True,
    "novos_treinamentos": 0, "modelos_substituidos": False,
    "modelo_em_uso_continua_baseline": True, "confianca_em_uso": .25,
    "auditoria": AUDITORIA, "dataset_sha256": DATASET_SHA,
    "validacao_nova": N_VALIDACAO, "reserva_futura": N_RESERVA_FUTURA,
    "reserva_futura_inferida": False, "test_home_inferido": False,
    "test_oi20k_inferido": False, "comparacao": COMPARACAO,
    "reproducao_home_val": REPRODUCAO, "integridade_conferida": True,
    "limites": [
        "Independência em relação aos dados conhecidos; pré-treinamentos não auditados.",
        "Home VAL histórica exploratória; observar a auditoria da proveniência.",
        "Open Images não garante cenas internas; revisar a adequação das cenas.",
        "Cobertura conservadora; observar suporte reduzido por classe.",
        "Bootstrap e indícios não estabelecem causas nem validam risco físico."],
    "proxima_etapa": "Revisar erros e definir controle Home versus Home+recorte alinhado, com atualizações comparáveis."}
core.save_json(PASTA_RESULTADOS/"resumo_diagnostico_11.json", RESUMO)
PACOTE_RESULTADOS = relatorios.package_reports(PASTA_RESULTADOS)
enviar_resultados("pacote final")
print("NOTEBOOK 11 CONCLUÍDO — DIAGNÓSTICO SEM TREINO; RESERVA FUTURA NÃO INFERIDA.")
print("Resultados:", PASTA_RESULTADOS)
print('Arquivo de resultados:', PACOTE_RESULTADOS.name)
print("Alternativa menor: resumo_diagnostico_11.json, comparacao_modelos.csv, bootstrap_pareado.csv e auditoria_baseline.json")
display(FileLink(str(PACOTE_RESULTADOS)))


## Referências e próximo experimento

- [Open Images: downloads e formatos](https://storage.googleapis.com/openimages/web/download_v7.html)
- [Open Images: cobertura parcial](https://storage.googleapis.com/openimages/web/evaluation.html)
- [CVDF: imagens de validação](https://github.com/cvdfoundation/open-images-dataset)
- [Separação entre treino, validação e teste](https://scikit-learn.org/stable/modules/cross_validation.html)

Após a inspeção, definiremos o treino de controle Home versus Home mais
um recorte alinhado. Este notebook não inicia esse treino.

Um manifesto original opcional deve vincular modelo_sha256 ao baseline e
conter listas train_image_sha256 e val_image_sha256. A ausência de interseção
só é interpretada como tal quando manifesto_treino_completo=true.
Um documento incompleto não prova ausência de exposição.
